# EX-056 — Janela 12

**B555 · Redes Neurais** · Aula **05** · Colab autocontido (EX-056 de 104)

## Enunciado

Com janela=12, quantos pares (X,y) restam?

## Como usar neste Colab

1. **Runtime → Executar tudo** (Run all)
2. Leia as explicações em markdown **antes** de cada célula de código
3. Olhe a **imagem** gerada e responda o enunciado
4. Não é necessário `git clone` — o código completo está abaixo

Espelho público (opcional): [`naubergois/b555-labs`](https://github.com/naubergois/b555-labs)


### O que este exercício treina
- **Aula 05** · `EX-056` · tipo `pratico`
- Dataset / figura: `janela_count`

### Como ler o código abaixo
1. A célula **Helpers** define seed, plots e funções reutilizáveis.
2. A célula **Exercício** carrega dados livres, calcula e **mostra a imagem**.
3. Leia os comentários `# >>>` no código — explicam cada passo.
4. Responda o enunciado olhando a figura / números impressos.

**Dica:** Siga o enunciado e interprete a figura gerada.


## 1) Instalar dependências

Pacotes livres: NumPy, pandas, scikit-learn, matplotlib.  
(No Colab, TensorFlow já costuma vir instalado quando o exercício de CNN precisar.)


In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib
print("deps OK")


## 2) Helpers (código compartilhado explicado)

Esta célula define:
- `SEED = 42` — reprodutibilidade
- `plot_fronteira`, `plot_loss`, `plot_sigmoid`, `plot_diagrama_neuronio` — imagens do exercício
- `titulo_ex` — cabeçalho visual no notebook

**Rode antes** da célula do exercício.


In [ ]:
# >>> HELPERS — funções compartilhadas (plots + seed)
# Seed fixa (42) = resultados reproduzíveis.
# plot_* geram as imagens didáticas do exercício.

#!/usr/bin/env python3
"""Helpers compartilhados dos exercícios B555 (plots + seed)."""
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
RNG = np.random.default_rng(SEED)
plt.rcParams.update({"figure.figsize": (6, 3.5), "axes.grid": True, "grid.alpha": 0.3})


def titulo_ex(ex_id: str, titulo: str) -> None:
    try:
        from IPython.display import Markdown, display

        display(Markdown(f"### {ex_id} — {titulo}"))
    except Exception:
        print(f"\n=== {ex_id} — {titulo} ===")


def show_img_title(ax, title: str) -> None:
    ax.set_title(title, fontsize=11)


def plot_sigmoid():
    z = np.linspace(-6, 6, 200)
    s = 1 / (1 + np.exp(-z))
    fig, ax = plt.subplots()
    ax.plot(z, s, lw=2, color="#0B6E4F")
    ax.axhline(0.5, ls="--", color="#888")
    ax.axvline(0, ls="--", color="#888")
    ax.scatter([0], [0.5], s=60, zorder=3, color="#C45C26")
    show_img_title(ax, "Sigmoid σ(z) — imagem do exercício")
    ax.set_xlabel("z")
    ax.set_ylabel("σ(z)")
    plt.tight_layout()
    plt.show()


def plot_fronteira(X, y, w, b, title="Fronteira de decisão"):
    fig, ax = plt.subplots()
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="#3D5A80", label="classe 0", alpha=0.7)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="#E07A5F", label="classe 1", alpha=0.7)
    xs = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)
    if abs(w[1]) > 1e-6:
        ys = -(w[0] * xs + b) / w[1]
        ax.plot(xs, ys, "k-", lw=2, label="fronteira")
    show_img_title(ax, title)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_loss(hist, title="Curva de loss"):
    fig, ax = plt.subplots()
    ax.plot(hist, color="#1B4965", lw=2)
    show_img_title(ax, title)
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    plt.tight_layout()
    plt.show()


def plot_diagrama_neuronio():
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 4)
    ax.axis("off")
    boxes = [
        (0.5, 2.5, "x₁"),
        (0.5, 1, "x₂"),
        (3.5, 1.75, "Σ w·x+b"),
        (6.2, 1.75, "φ(·)"),
        (8.5, 1.75, "ŷ"),
    ]
    for x, y, t in boxes:
        ax.add_patch(
            plt.Rectangle((x, y), 1.4, 0.8, fill=True, color="#E8F1F2", ec="#1B4965", lw=2)
        )
        ax.text(x + 0.7, y + 0.4, t, ha="center", va="center", fontsize=10)
    ax.annotate("", xy=(3.5, 2.1), xytext=(1.9, 2.9), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(3.5, 1.9), xytext=(1.9, 1.4), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(6.2, 2.15), xytext=(4.9, 2.15), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(8.5, 2.15), xytext=(7.6, 2.15), arrowprops=dict(arrowstyle="->"))
    show_img_title(ax, "Diagrama do neurônio — imagem do exercício")
    plt.tight_layout()
    plt.show()


# Atalhos usados pelos exercícios
print("Helpers OK · SEED =", SEED)


## 3) Exercício EX-056 — código completo

Abaixo está o **mesmo código** de `ex_056.py`, já embutido e comentado.

O que acontece, em ordem:
1. Carrega o **dataset livre** (sklearn / Keras / CSV público)
2. Calcula o que o enunciado pede
3. **Plota a imagem** didática
4. Imprime números para você responder

<details><summary>Gabarito curto (só depois de tentar)</summary>

_interprete a figura e os números impressos_

</details>


In [ ]:
# >>> Código completo do exercício (rode a célula inteira)
# >>> Não precisa clonar repositório: tudo está neste Colab.

titulo_ex("EX-056", "Janela 12")
import pandas as pd
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
try:
    df = pd.read_csv(url)
    col = [c for c in df.columns if c.lower() != "month"][0]
    y = df[col].astype(float).values
except Exception:
    t = np.arange(144)
    y = 100 + 0.5 * t + 20 * np.sin(t / 6)
janela = 12
n = len(y) - janela
print("pares", n)
fig, ax = plt.subplots()
ax.bar(["n_pares"], [n])
show_img_title(ax, f"janela={janela}")
plt.tight_layout(); plt.show()


## 4) Fecho

Você concluiu **EX-056** com código explicado neste Colab.

- Próximo exercício: veja o [índice](https://github.com/naubergois/b555-labs/blob/main/labs/colabs/INDEX.md)
- Frase-guia: **a rede calcula; nós decidimos.**
